# <strong>Worms: Modeling the Spread</strong>

In this exercise, we are going to use the PAWS simulator developed by S. Wei and Dr. J. Mirkovic <a href="http://portal.acm.org/citation.cfm?id=1070162">[1]</a> using the DETER testbed. PAWS is a discrete time packet-level simulator which simulates a realistic Internet model and the background traffic load, enabling investigation of possible congestion effects and sufferings of legitimate traffic during a worm attack. PAWS supports various user-customizable parameters that enable testing of different worm characteristics, host and network diversity models. Before you do this experiment, you should read carefully the reference papers <a href="http://www.isi.edu/%7Emirkovic/publications/trident09.pdf">[2]</a>, <a href="http://portal.acm.org/citation.cfm?id=948197">[3]</a>, <a href="http://cseweb.ucsd.edu/%7Esavage/papers/OSDI04.pdf">[8]</a>, <a href="http://www.cse.lehigh.edu/%7Echuah/publications/paper-ijsn-revised.pdf">[9]</a>. In this exercise, you will replicate an experiment which was carried out by the authors in [3] to evaluate how their proposed host-based dynamic quarantine system can help to contain Internet worm propagation. Table 1 lists the original experiment settings in <a href="http://portal.acm.org/citation.cfm?id=948197">[3]</a> and the configuration and customization of PAWS used in <a href="http://www.isi.edu/%7Emirkovic/publications/trident09.pdf">[2]</a> to match the settings in <a href="http://portal.acm.org/citation.cfm?id=948197">[3]</a>. 

### <strong>Required Reading</strong>
- <a href="http://en.wikipedia.org/wiki/SQL_slammer_%28computer_worm%29">Short summary of SlammerWorm attack on Wikipedia</a>
- <a href="http://en.wikipedia.org/wiki/Witty_worm">Short summary of WiityWorm attack on Wikipedia</a>
- <a href="http://www.tcpdump.org/tcpdump_man.html">Tcpdump's man page</a>

### Step 0: Starting the Lab

Click the button to begin creating the experiment.

<strong>Note:</strong> If your buttons are not displaying, click on the <img width='20px' height='20px' style='margin-left: 1px;' src='resources/fast_forward.png'> icon at the top of your notebook to render all widgets.

In [2]:
# Click on the button below to start your lab.

# Before running imports, check to see if the extensions are installed.
# Installing necessary imports first. Rest will follow.
import os
import subprocess
import sys
from IPython.display import display, HTML
from IPython import get_ipython
import re
import threading
import queue
import time
import logging
import shutil
import tempfile
import tarfile
from pathlib import Path

# This will ensure that the require extensions are installed correctly.
# Sometimes, a SPHERE update will remove necessary extensions for the notebooks.
from resources.reinstall_notebooks import installation

# Now, checking to make sure that the extension is installed before we continue imports.
installation()

# ipywidgets exists. We will carry out with the rest of the imports.
import ipywidgets as widgets

# The lab name.
labname = "worm"

# Adding the "resources/" directory so that we can import the start.py file.
module_dir = os.path.join(os.getcwd(), 'resources')
if module_dir not in sys.path:
    sys.path.append(module_dir)

# Importing the prepare_lab function.
from functions import *

# Required for Step 5 to work.
step1Complete = step2Complete = step3Complete = runAllSteps = False

def setup_lab():
    try:
        # Check if ipywidgets is imported.
        if 'widgets' not in globals():
            raise ImportError("Jupyter Widgets not imported correctly.")
    except ImportError as e:
        display(HTML(
            f"<div style='color: red;'>"
            f"Jupyter Widgets was not imported correctly. Error: {e}<br>"
            "Please re-run <code>install_notebooks.sh</code>, <u>refresh your browser tab</u>, then try again."
            "</div>"
        ))
        raise e

    # Defining UI.
    output0 = widgets.Output()
    startButton = widgets.Button(description="Start Lab")
    

    # Defining the button handler.
    def on_start_clicked(b):
        prepare_lab(labname, output0)

    startButton.on_click(on_start_clicked)
    display(startButton, output0)

setup_lab()

Jupyter Widgets isn't installed. This is possibly due to an update rolled out by SPHERE. The notebook will revert your installations. Please wait.

Running the installation. If see a pop-up window appear, click OVERWRITE. Please wait until the notebook fixes your installation...

Please click the fast-forward icon (>>) at the top of the notebooks, and try re-compiling again.



<hr>

If you previously stopped your lab by using the "Stop Lab" button at the bottom of the notebook, you may restore your progress below by clicking "Load Lab". <u>You do not have to load your lab if you signed out, closed your notebook, or exited your node(s) or XDC by using ```exit```.</u>

In [2]:
# Click the button below to load your lab.
def loadlab(b):
    load_lab(labname, output0_2)

# Creating the button.
loadButton = widgets.Button(description="Load Lab")

# Creating an output area.
output0_2 = widgets.Output()

# Run the command on click.
loadButton.on_click(loadlab)

# Display the output.
display(loadButton, output0_2)

Button(description='Load Lab', style=ButtonStyle())

Output()

### Step 1: Preparing Your Attack

There are two nodes that you will access on this lab: `node-0` and `node-1`.

The `/tmp/node-0/paws` on `node-0` folder contains the following files:

- `ASLinks.dat` - a file which contains all the inter-AS links with corresponding bandwidth values assigned
- `BGPAtom2AS.dat` - a file which maps BGP atoms to the owner ASes.
- `IPRangeTable.dat` - a file which contains all the IP ranges with their owner BGP atoms and ASes. 

Similarly, the folder `/tmp/node-1/paws` on `node-1` contains the same files. 

<strong>Your task:</strong> Follow these steps.

1. Log onto `node-0`
2. In the `/tmp/node-0/paws` directory run `sudo make`.
3. Log onto `node-1`
4. In the `/tmp/node-1/paws` directory run `sudo make`.
5. Run: `sudo curl https://steel.isi.edu/paws_RT.dat.gz -o paws_RT.dat.gz`. Then, run `sudo gunzip paws_RT.dat.gz`. 

Upon clicking "Check Work", the notebook will check to make sure that you have ran `sudo make` in each node. It will also check to make sure that `paws_RT.dat.gz` has been unzipped, and exists on `node-1`.

In [3]:
# Click the button below to check your work.
def step_1():
    # Important variables that must be accessed outside of this function.
    global step1Complete, result

    with output1:
        output1.clear_output()
        display(HTML("<span><img width='12px' height='12px' style='margin-left: 3px;' src='resources/loading.gif'></span>"))
    
    result = subprocess.run([
        'ssh', '-o', 'StrictHostKeyChecking=no', '-i', 
        '/home/USERNAME_GOES_HERE/.ssh/merge_key', 'USERNAME_GOES_HERE@node-0', 
        '/home/.checker/step_1.py'])

    if (result.returncode == 0):
        output1.clear_output()
        with output1:
            display(HTML("<span style='color: green;'>Success! You may continue onto the next step.</span>"))
            step1Complete = True

    elif (result.returncode == 1):
        output1.clear_output()
        with output1:
            display(HTML("<span style='color: red;'>You have not ran <code>sudo make</code> on <code>node-0</code> yet.</span>"))
            step1Complete = False

    elif (result.returncode == 2):
        output1.clear_output()
        with output1:
            display(HTML("<span style='color: red;'>You have not ran <code>sudo make</code> on <code>node-1</code> yet.</span>"))
            step1Complete = False

    elif (result.returncode == 3):
        output1.clear_output()
        with output1:
            display(HTML("<span style='color: red;'>You have not ran <code>sudo gunzip paws_RT.dat.gz</code> on <code>node-1</code> yet. Make sure to run the <code>curl</code> before running this unzip command. Ensure you're on the <code>node-1</code> node.</span>"))
            step1Complete = False

    elif (result.returncode == 4):
        output1.clear_output()
        with output1:
            display(HTML("<span style='color: red;'>There was an error running this step. Please contact your instructor/TA.</span>"))
            step1Complete = False

def check_step_1(b):
    if (warn_student(labname)):
        output1.clear_output()
        with output1:
            display(HTML("<span style='color: red;'><strong>WARNING:</strong> You have an autosaved lab that you have not yet loaded. If you would like to load your progress, click \"Load Lab\" at the top of the notebook. Otherwise, clicking on this button again will assume you're restarting the lab!</span>"))
    else:
        step_1()

        # Auto-save.
        if (not runAllSteps):
            trigger_save(labname, "1", result.returncode)

# Creating the button.
button = widgets.Button(description="Check Work")

# Creating an output area.
output1 = widgets.Output()

# Run the command on click.
button.on_click(check_step_1)

# Display the output.
display(button, output1)

Button(description='Check Work', style=ButtonStyle())

Output()

### Step 2: Performing Your Attack

<strong>Your task:</strong> In `paws` folder, run `./paws_server > ~/log` on `node-0` and `./paws_client > ~/log` on `node-1`. Code may run slowly toward the end and it may take about 1-2 hours to finish.

Analyze the data that's collected in each log file.

This step will check to make sure that `~/log` exists on each node.

<span style="color: green"><strong><img src="resources/idea.png" style="width: 12px"> Tip:</strong></span> SPHERE allows you to create multiple tabs, similar to a browser. Use this to your advantage by opening an instance of `node-0` and `node-1`. Then, you can perform both attacks simultaneously.

In [4]:
# Click the button below to check your work.
def step_2():
    global step2Complete, result

    with output2:
        output2.clear_output()
        display(HTML("<span><img width='12px' height='12px' style='margin-left: 3px;' src='resources/loading.gif'></span>"))
    
    result = subprocess.run([
        'ssh', '-o', 'StrictHostKeyChecking=no', '-i', 
        '/home/USERNAME_GOES_HERE/.ssh/merge_key', 'USERNAME_GOES_HERE@node-0', 
        '/home/.checker/step_2.py'])

    if (result.returncode == 0):
        output2.clear_output()
        with output2:
            display(HTML("<span style='color: green;'>Success! You may continue onto the next step.</span>"))
            step2Complete = True

    elif (result.returncode == 1):
        output2.clear_output()
        with output2:
            display(HTML("<span style='color: red;'><code>~/log</code> doesn't exist on <code>node-0</code>. Ensure that you've ran your attack with the command above.</span>"))
            step2Complete = False

    elif (result.returncode == 2):
        output2.clear_output()
        with output2:
            display(HTML("<span style='color: red;'><code>~/log</code> doesn't exist on <code>node-1</code>. Ensure that you've ran your attack with the command above.</span>"))
            step2Complete = False

    elif (result.returncode == 3):
        output2.clear_output()
        with output2:
            display(HTML("<span style='color: red;'>There was an error checking this step. Please contact your professor/TA.</span>"))
            step2Complete = False

def check_step_2(b):
    if (warn_student(labname)):
        output2.clear_output()
        with output2:
            display(HTML("<span style='color: red;'><strong>WARNING:</strong> You have an autosaved lab that you have not yet loaded. If you would like to load your progress, click \"Load Lab\" at the top of the notebook. Otherwise, clicking on this button again will assume you're restarting the lab!</span>"))
    else:
        step_2()

        # Auto-save.
        if (not runAllSteps):
            trigger_save(labname, "2", result.returncode)

# Creating the button.
button = widgets.Button(description="Check Work")

# Creating an output area.
output2 = widgets.Output()

# Run the command on click.
button.on_click(check_step_2)

# Display the output.
display(button, output2)

Button(description='Check Work', style=ButtonStyle())

Output()

### Step 3: Replicating the Slammer Worm Experiment

Next, you need to replicate the Slammer Worm experiment (refer to the required reading at the top of this notebook). Note that this choice can be made in `client.c` in the `paws` directory by choosing either the `#define SLAMMER_WORM` or `WITTY_WORM` line and recompiling using `make`. 

<strong>Your task:</strong> Re-compile your programs with `sudo make`, then create a NEW log with these changes, called `~/log_2` on each node. 

<span style="color: green"><strong><img src="resources/idea.png" style="width: 12px"> Tips:</strong></span>
- If you run into a <strong>segmentation fault</strong>, it may be caused by an incomplete `paws_RT.dat` file. A complete version of that file should be about 645 Mbytes long.
- Always re-run `make` in the `paws` directory if any `.c` or `.h` file is modified. 

In [5]:
# Click the button below to check your work.
def step_3():
    global step3Complete, result

    with output3:
        output3.clear_output()
        display(HTML("<span><img width='12px' height='12px' style='margin-left: 2px;' src='resources/loading.gif'></span>"))
    
    result = subprocess.run([
        'ssh', '-o', 'StrictHostKeyChecking=no', '-i', 
        '/home/USERNAME_GOES_HERE/.ssh/merge_key', 'USERNAME_GOES_HERE@node-0', 
        '/home/.checker/step_3.py'])

    if (result.returncode == 0):
        output3.clear_output()
        with output3:
            display(HTML("<span style='color: green;'>Success! You may continue onto the next step.</span>"))
            step3Complete = True

    elif (result.returncode == 1):
        output3.clear_output()
        with output3:
            display(HTML("<span style='color: red;'><code>~/log_2</code> doesn't exist on <code>node-0</code>. Ensure that you've ran your attack with the command above.</span>"))
            step3Complete = False

    elif (result.returncode == 2):
        output3.clear_output()
        with output3:
            display(HTML("<span style='color: red;'><code>~/log_2</code> doesn't exist on <code>node-1</code>. Ensure that you've ran your attack with the command above.</span>"))
            step3Complete = False

    elif (result.returncode == 3):
        output3.clear_output()
        with output3:
            display(HTML("<span style='color: red;'>There was an error checking this step. Please contact your professor/TA.</span>"))
            step3Complete = False

def check_step_3(b):
    if (warn_student(labname)):
        output3.clear_output()
        with output3:
            display(HTML("<span style='color: red;'><strong>WARNING:</strong> You have an autosaved lab that you have not yet loaded. If you would like to load your progress, click \"Load Lab\" at the top of the notebook. Otherwise, clicking on this button again will assume you're restarting the lab!</span>"))
    else:
        step_3()

        # Auto-save.
        if (not runAllSteps):
            trigger_save(labname, "3", result.returncode)

# Creating the button.
button = widgets.Button(description="Check Work")

# Creating an output area.
output3 = widgets.Output()

# Run the command on click.
button.on_click(check_step_3)

# Display the output.
display(button, output3)

Button(description='Check Work', style=ButtonStyle())

Output()

### Step 4: Analyze Your Results

<strong>Your task:</strong> Plot two curves similar to Figure 1 in <a href="https://www.isi.edu/people-mirkovic/wp-content/uploads/sites/52/2023/10/trident09.pdf">Tools for Worm Experimentation on the DETER Testbed</a>. 

The <u>first plot</u> should show how the number of infected hosts varies with time for an original system and a quarantined system. 

The <u>second plot</u> should show how the number of infected hosts and the number of quarantined hosts vary with time. 

In your submitted report, you should first describe briefly how Slammer worm spreads in the Internet and summarize how the proposed host-based dynamic quarantine system works. Then, include your plots and discuss the impacts of varying some of the configuration parameters e.g. the quarantine rate. 

<strong>There is no auto-grader for this step.</strong>

### Step 5: Generate Your Submission

Once you have written your report from Step 4, you may submit your work. Here's what is required for you to submit:
1. Your written report
2. Your modified programs (`client.c` in your `node-0` and `node-1` devices)

The report should describe how Slammer worm spreads and how the proposed host-based dynamic quarantine system works. It should also include your plots and your discussions on the impacts of varying some of the configured parameters e.g. the quarantine rate. 

<strong>Your task:</strong> Click the button below to generate a tarball named `USERNAME_GOES_HERE_worms.tar.gz`, which contains your two modified programs. <strong>To automatically include your report in the submission</strong>, upload a file to the sidebar of your XDC called `USERNAME_GOES_HERE_report.docx` or `USERNAME_GOES_HERE_report.pdf`. Clicking on the button will automatically include this file in your submission.

<strong>If you do not include this file</strong>, your submission will still be generated. However, make sure that you manually include your report before submitting it to your instructor.

<strong>This step will not run until you have completed Steps 1-3.</strong>

In [6]:
# Click the button below to check your work.
def step_5():
    global step1Complete, step2Complete, step3Complete, step5Complete, result

    if (not step1Complete or not step2Complete or not step3Complete):
        output5.clear_output()
        with output5:
            display(HTML("<span style='color: red;'>Please complete Steps 1-3 before running Step 5.</span>"))
            step5Complete = False
            return

    with output5:
        output5.clear_output()
        display(HTML("<span>Generating your final submissions. Please wait a few seconds... <img width='12px' height='12px' style='margin-left: 2px;' src='resources/loading.gif'></span>"))

    home = Path.home()
    temp_dir = tempfile.mkdtemp()
    
    # Define target files and their destination in temp_dir.
    file_paths = [
        ("node-0:/tmp/node-0/paws/client.c", os.path.join(temp_dir, "client_node0.c")),
        ("node-1:/tmp/node-1/paws/client.c", os.path.join(temp_dir, "client_node1.c"))
    ]

    # Copy files from remote nodes using scp.
    for remote, local in file_paths:
        result = subprocess.run(
            ['scp', '-o', 'StrictHostKeyChecking=no', remote, local],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )

    # Check for optional report files in the user's home directory.
    docx_path = home / "USERNAME_GOES_HERE_report.docx"
    pdf_path = home / "USERNAME_GOES_HERE_report.pdf"
    doc_found = False

    if docx_path.exists():
        shutil.copy(docx_path, temp_dir)
        doc_found = True
    elif pdf_path.exists():
        shutil.copy(pdf_path, temp_dir)
        doc_found = True

    # Create tarball entirely within the temp directory.
    tar_name = "USERNAME_GOES_HERE_worm.tar.gz"
    tar_path = os.path.join(temp_dir, tar_name)
    with tarfile.open(tar_path, "w:gz") as tar:
        for file_name in os.listdir(temp_dir):
            full_path = os.path.join(temp_dir, file_name)
            # Don't add the tarball itself into the tar
            if full_path != tar_path:
                tar.add(full_path, arcname=file_name)

    # Move the tarball to the home directory.
    final_tar_path = home / tar_name
    shutil.move(tar_path, final_tar_path)

    # Output messages based on step completion.
    if doc_found:
        output5.clear_output()
        with output5:
            display(HTML("<span style='color: green;'>Your submission (USERNAME_GOES_HERE_worm.tar.gz) was generated WITH your report. It is located in the sidebar of your XDC.</span>"))
            step5Complete = True

    else:
        output5.clear_output()
        with output5:
            display(HTML("<span style='color: green;'>Your submission (USERNAME_GOES_HERE_worm.tar.gz) was generated WITHOUT your report. It is located in the sidebar of your XDC. Ensure that you include your report in your final submission!</span>"))
            step5Complete = True

def check_step_5(b):
    if (warn_student(labname)):
        output5.clear_output()
        with output5:
            display(HTML("<span style='color: red;'><strong>WARNING:</strong> You have an autosaved lab that you have not yet loaded. If you would like to load your progress, click \"Load Lab\" at the top of the notebook. Otherwise, clicking on this button again will assume you're restarting the lab!</span>"))
    else:
        step_5()

        # Auto-save.
        if (not runAllSteps):
            if ("result" in locals()):
                trigger_save(labname, "5", "0")

            else:
                trigger_save(labname, "5", "1")

# Creating the button.
button = widgets.Button(description="Check Work")

# Creating an output area.
output5 = widgets.Output()

# Run the command on click.
button.on_click(check_step_5)

# Display the output.
display(button, output5)

Button(description='Check Work', style=ButtonStyle())

Output()

## <strong>Grading</strong>

To check your overall grade, click on the button below.

In [39]:
# Click the button below to check your overall grade.
steps_to_check = [step_1, step_2, step_3, step_5]   

# Function to calculate grade after refreshing the cell.
def calculate_grade(b):
    # To not auto-save at each step.
    global runAllSteps
    runAllSteps = True

    with gradeOutput:
        gradeOutput.clear_output()
        display(HTML("<span>Testing all steps. Please wait.</span> \
            <span><img width='12px' height='12px' style='margin-left: 3px;' src='resources/loading.gif'></span>"))
    
    # Required for checking the boolean values.
    for func in steps_to_check:
        func()

    steps = [step1Complete, step2Complete, step3Complete, step5Complete]
    output = ""
    stepsCorrect = 0

    # Adding one because Step 4 is not graded, and is not counted for within "steps".
    numOfSteps = len(steps) + 1

    for i in range(1, numOfSteps):
        if i == 4:
            output += "<div style='color: orange;'>Step 4 is not automatically graded.</div>"
            
        if steps[i - 1]:
            stepsCorrect += 1
            # Hardcoding this. Will be more difficult to grade if future steps are added.
            if (i == 4):
                output += "<div style='color: green;'>Step 5 is complete.</div>"

            else:
                output += "<div style='color: green;'>Step " + str(i) + " is complete.</div>"

        else:
            # Hardcoding this. Will be more difficult to grade if future steps are added.
            if (i == 4):
                output += "<div style='color: red;'>Step 5 is incomplete.</div>"

            else:
                output += "<div style='color: red;'>Step " + str(i) + " is incomplete.</div>"
                
    # Removing the extra step that was added (Step 4) since it should not be counted with the overall grade.
    output += "<div style='color: black;'>You have " + str(stepsCorrect) + " out of " + str(numOfSteps - 1) + " steps completed.</div>"

    with gradeOutput:
        gradeOutput.clear_output()
        display(HTML(output))

    # Disable this boolean so that saves will work again.
    runAllSteps = False

# Create a button to refresh the cell and another to calculate grade.
grade_button = widgets.Button(description="Calculate Grade")

# Link buttons to functions.
grade_button.on_click(calculate_grade)

# Output area.
gradeOutput = widgets.Output()

# Display the buttons and output.
display(grade_button, gradeOutput)

Button(description='Calculate Grade', style=ButtonStyle())

Output()

### Stopping the Lab

Once you are done with the lab, click on the "Stop Lab" button below. <strong>This will delete your activation, which will delete all of the lab's resources.</strong> Your progress is saved automatically in ```saves/``` within the sidebar of your XDC. You may load this lab in the future by clicking "Load Lab" at the top.

In [8]:
# Click the button below to stop the experiment.
def stoplab(button):
    stop_lab(labname, confirm, stop_output)

# Creating the button.
stopButton = widgets.Button(description="Stop Lab")

# Create a confirmation check.
confirm = widgets.Checkbox(
    value=False,
    description='Confirm',
    disabled=False,
    indent=False
)

# Creating an output area.
stop_output = widgets.Output()

# Run the command on click.
stopButton.on_click(stoplab)

# Display the output.
display(confirm, stopButton, stop_output)

Checkbox(value=False, description='Confirm', indent=False)

Button(description='Stop Lab', style=ButtonStyle())

Output()